# Wind & Solar Capacity Predictor

Target => val_geracaoverificada

In [ ]:
import pandas as pd
import numpy as np
from os import listdir
from pathlib import Path

Agregando os dados extraidos da ONS:

In [ ]:
df = pd.concat(
    [pd.read_parquet(f"./database/{file}") for file in listdir("./database") if file.endswith('.parquet')],
    ignore_index=True
)

Ajustando a tipagem das colunas do DF de acordo com a documentação:

In [ ]:
float_columns = [
    "val_latitudesecoletora",
    "val_longitudesecoletora",
    "val_latitudepontoconexao",
    "val_longitudepontoconexao",
    "val_geracaoverificada",
    "val_capacidadeinstalada",
    "val_fatorcapacidade"
]

df[float_columns] = df[float_columns].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

df = df.drop("val_geracaoprogramada", axis=1)

df["din_instante"] = pd.to_datetime(df["din_instante"])

In [ ]:
print(df.sort_values(by="din_instante").iloc[1]) # Encontra o registro mais antigo

Lê os dados metereológicos extraídos do (OpenMeteo)[https://open-meteo.com/] a partir de 2009 até 2026 (Novembro e Dezembro excluidos) com as coordenadas das capitais das unidades federativas do Brasil salvando todas as informações diárias disponíveis (inclusive as "opcionais"):

In [ ]:
def _num(valor):
    """Converte '1159,54' -> 1159.54; qualquer valor inválido vira NaN."""
    return pd.to_numeric(str(valor).replace(",", "."), errors="coerce")

def ler_inmet(caminho):
    # 1) Metadados (8 primeiras linhas)
    meta = {}
    with open(caminho, encoding="latin-1") as f:
        for _ in range(8):
            chave, _, valor = f.readline().strip().partition(";")
            meta[chave.rstrip(":")] = valor.strip()

    # 2) Dados
    df = pd.read_csv(
        caminho,
        sep=";",
        skiprows=8,
        decimal=",",
        encoding="latin-1",
        na_values=["-9999"],
        index_col=False,
    )
    df = df.loc[:, ~df.columns.str.startswith("Unnamed")]

    df["REGIAO"] = meta.get("REGIÃO")
    df["UF"] = meta.get("UF")
    df["ESTACAO"] = meta.get("ESTAÇÃO")
    df["CODIGO_WMO"] = meta.get("CODIGO (WMO)")
    df["LATITUDE"] = _num(meta.get("LATITUDE"))
    df["LONGITUDE"] = _num(meta.get("LONGITUDE"))
    df["ALTITUDE"] = _num(meta.get("ALTITUDE"))
    df["DATA_FUNDACAO"] = pd.to_datetime(
        meta.get("DATA DE FUNDAÇÃO (YYYY-MM-DD)"), errors="coerce"
    )

    return df

pasta = Path("./database/INMET/2017")

df_weather = pd.concat(
    [ler_inmet(pasta / f) for f in listdir(pasta) if f.lower().endswith(".csv")],
    ignore_index=True,
)

In [ ]:
float_columns = [
    "PRECIPITAÇÃO TOTAL, HORÁRIO (mm)",
    "PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)",
    "PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)",
    "PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)",
    "RADIACAO GLOBAL (KJ/m²)",
    "TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)",
    "TEMPERATURA DO PONTO DE ORVALHO (°C)",
    "TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)",
    "TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)",
    "UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)",
    "UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)",
    "UMIDADE RELATIVA DO AR, HORARIA (%)",
    "VENTO, DIREÇÃO HORARIA (gr) (° (gr))",
    "VENTO, RAJADA MAXIMA (m/s)",
    "VENTO, VELOCIDADE HORARIA (m/s)",
    "LATITUDE",
    "LONGITUDE",
    "ALTITUDE",
]

category_columns = ["REGIAO", "UF", "ESTACAO", "CODIGO_WMO"]

df_weather[float_columns] = df_weather[float_columns].apply(
    lambda col: pd.to_numeric(col, errors="coerce")
)

hora = (
    df_weather["HORA (UTC)"]
    .astype(str)
    .str.replace(" UTC", "", regex=False)
    .str.replace(":", "", regex=False)
    .str.zfill(4)
)

df_weather["DATETIME"] = pd.to_datetime(
    df_weather["DATA (YYYY-MM-DD)"].astype(str) + " " + hora,
    format="%Y-%m-%d %H%M",
    errors="coerce",
    utc=True,
)
df_weather["DATA_FUNDACAO"] = pd.to_datetime(df_weather["DATA_FUNDACAO"], errors="coerce")

df_weather = df_weather.drop(columns=["DATA (YYYY-MM-DD)", "HORA (UTC)"])

df_weather[category_columns] = df_weather[category_columns].astype("category")

Join dos Datasets:

In [ ]:
import numpy as np
import pandas as pd
from sklearn.neighbors import BallTree

# O INMET aqui é só de 2017, então filtra o df para o mesmo ano
df_2017 = df[df["din_instante"].dt.year == 2017].copy()

# ------------------------------------------------------------
# Prepara o weather (cria DATETIME se ainda não existir)
# ------------------------------------------------------------
weather = df_weather.copy()

if "DATETIME" not in weather.columns:
    hora = (
        weather["HORA (UTC)"]
        .astype(str)
        .str.replace(" UTC", "", regex=False)
        .str.replace(":", "", regex=False)
        .str.zfill(4)
    )
    weather["DATETIME"] = pd.to_datetime(
        weather["DATA (YYYY-MM-DD)"].astype(str) + " " + hora,
        format="%Y-%m-%d %H%M",
        errors="coerce",
        utc=True,
    )
    weather = weather.drop(columns=["DATA (YYYY-MM-DD)", "HORA (UTC)"])

weather["CODIGO_WMO"] = weather["CODIGO_WMO"].astype(str)
weather = weather.drop_duplicates(["CODIGO_WMO", "DATETIME"])

# ------------------------------------------------------------
# 1) Usina -> estação mais próxima
# ------------------------------------------------------------
plantas = (
    df_2017[["id_ons", "val_latitudesecoletora", "val_longitudesecoletora"]]
    .dropna()
    .drop_duplicates("id_ons")
    .reset_index(drop=True)
)

estacoes = (
    weather[["CODIGO_WMO", "LATITUDE", "LONGITUDE"]]
    .dropna()
    .drop_duplicates("CODIGO_WMO")
    .reset_index(drop=True)
)

tree = BallTree(
    np.radians(estacoes[["LATITUDE", "LONGITUDE"]].to_numpy()),
    metric="haversine",
)
dist, idx = tree.query(
    np.radians(plantas[["val_latitudesecoletora", "val_longitudesecoletora"]].to_numpy()),
    k=1,
)

plantas["CODIGO_WMO"] = estacoes["CODIGO_WMO"].to_numpy()[idx[:, 0]]
plantas["dist_estacao_km"] = dist[:, 0] * 6371  # raio da Terra em km

# ------------------------------------------------------------
# 2) Alinha o tempo (ONS = horário de Brasília, INMET = UTC)
# ------------------------------------------------------------
df_2017["DATETIME"] = (
    (df_2017["din_instante"] + pd.Timedelta(hours=3))
    .dt.tz_localize("UTC")
    .astype("datetime64[ns, UTC]")
)
weather["DATETIME"] = weather["DATETIME"].astype("datetime64[ns, UTC]")

# ------------------------------------------------------------
# 3) Join
# ------------------------------------------------------------
df_2017 = df_2017.merge(
    plantas[["id_ons", "CODIGO_WMO", "dist_estacao_km"]],
    on="id_ons",
    how="left",
)

weather = weather.rename(columns={
    "LATITUDE": "LAT_ESTACAO",
    "LONGITUDE": "LON_ESTACAO",
    "ALTITUDE": "ALT_ESTACAO",
})

df_final = df_2017.merge(
    weather,
    on=["CODIGO_WMO", "DATETIME"],
    how="left",
    validate="many_to_one",
)

In [ ]:
print(df_final.iloc[67])

Consulta SQL 1)

Consulta SQL 2)

Consulta SQL 3)

Consulta SQL 4)

Consulta SQL 5)

# Modelagem:

## Baseline:

## Modelos Obrigatórios: Random Forest e SVR

### Random Forest:

### SVR:

## Gradient Boosting:

## TabPFN: